# HAIO 2025 - Egyetemi Felvételi Megoldás

**Magyar MI Diákolimpia - Online Válogató 2025**

Feladat: Egyetemi felvételi eredmény előrejelzése (bináris klasszifikáció)  
Kiértékelési metrika: **ROC-AUC**  
Kaggle verseny: https://www.kaggle.com/competitions/magyar-mi-diakolimpia-online-valogato-2025

## 0. Telepítés és importok

In [ ]:
!pip install lightgbm xgboost geopandas --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, classification_report
from sklearn.model_selection import GridSearchCV

import xgboost as xgb
import lightgbm as lgb

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

RANDOM_STATE = 42
print('Importok sikeresek!')

## 1. Adatfeltárás (EDA)

### 1.1 Adatok betöltése

In [ ]:
# Adatok betöltése
train = pd.read_csv('../adatok/train.csv')
test = pd.read_csv('../adatok/test.csv')

print(f'Train méret: {train.shape}')
print(f'Test méret:  {test.shape}')
print(f'\nOszlopok ({len(train.columns)} db):')
print(train.columns.tolist())

In [ ]:
train.head(10)

In [ ]:
train.dtypes

In [ ]:
train.describe()

### 1.2 Hiányzó értékek és -1 vizsgálata

In [ ]:
# Valódi NaN értékek
print('=== Valódi NaN értékek ===')
print(train.isnull().sum())

print('\n=== -1 értékek száma oszloponként ===')
# A -1 értékek "hiányzó/nem alkalmazható" jelölők (pl. nem vett fel tantárgyat)
minus_one_counts = (train == -1).sum()
print(minus_one_counts[minus_one_counts > 0])

### 1.3 Célváltozó eloszlása

In [ ]:
TARGET = 'Felvételi Eredmény'

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Oszlopdiagram
target_counts = train[TARGET].value_counts()
colors = ['#e74c3c', '#2ecc71']
axes[0].bar(target_counts.index.astype(int), target_counts.values, color=colors)
axes[0].set_xlabel('Felvételi Eredmény')
axes[0].set_ylabel('Darabszám')
axes[0].set_title('Célváltozó eloszlása')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Elutasítva (0)', 'Felvéve (1)'])

for i, v in enumerate(target_counts.values):
    axes[0].text(target_counts.index[i], v + 30, str(v), ha='center', fontweight='bold')

# Kördiagram
axes[1].pie(target_counts.values, labels=['Elutasítva (0)', 'Felvéve (1)'],
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Célváltozó arányok')

plt.tight_layout()
plt.show()

print(f'Felvételi arány: {train[TARGET].mean():.3f}')

### 1.4 Fontos jellemzők vizualizációja

In [ ]:
# Osztályzatok eloszlása a célváltozó szerint
osztalyzatok = ['Osztályzat_9', 'Osztályzat_10', 'Osztályzat_11', 'Osztályzat_12']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, col in enumerate(osztalyzatok):
    ax = axes[i // 2][i % 2]
    for label, color in zip([0, 1], colors):
        subset = train[train[TARGET] == label][col]
        ax.hist(subset, bins=20, alpha=0.6, color=color,
                label=f'Eredmény={int(label)}')
    ax.set_title(col)
    ax.set_xlabel('Osztályzat')
    ax.set_ylabel('Darabszám')
    ax.legend()

plt.suptitle('Osztályzatok eloszlása felvételi eredmény szerint', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Érettségi tárgyak pontszámai
tantargyak = ['Történelem', 'Matematika', 'Magyar Nyelv és Irodalom',
              'Informatika', 'Biológia', 'Fizika', 'Angol', 'Német']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, col in enumerate(tantargyak):
    ax = axes[i // 4][i % 4]
    # Csak a nem -1 értékeket ábrázoljuk
    for label, color in zip([0, 1], colors):
        subset = train[(train[TARGET] == label) & (train[col] != -1)][col]
        ax.hist(subset, bins=25, alpha=0.6, color=color,
                label=f'Eredmény={int(label)}')
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Érettségi tantárgyak pontszámai felvételi eredmény szerint', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Korrelációs hőtérkép a numerikus oszlopokra
numerikus_oszlopok = train.select_dtypes(include=[np.number]).columns.tolist()
# ID-t kihagyjuk
numerikus_oszlopok = [c for c in numerikus_oszlopok if c != 'ID']

corr = train[numerikus_oszlopok].corr()

fig, ax = plt.subplots(figsize=(18, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            annot=True, fmt='.2f', square=True, linewidths=0.5,
            ax=ax, annot_kws={'size': 7})
ax.set_title('Korrelációs hőtérkép', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Célváltozóval való korreláció
target_corr = corr[TARGET].drop(TARGET).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
bar_colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in target_corr.values]
target_corr.plot(kind='barh', ax=ax, color=bar_colors)
ax.set_title('Korreláció a Felvételi Eredménnyel', fontsize=14)
ax.set_xlabel('Pearson-korreláció')
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

### 1.5 Vármegye szerinti vizualizáció

In [ ]:
# Vármegye szerinti felvételi arány
varmegye_stats = train.groupby('Vármegye')[TARGET].agg(['mean', 'count']).reset_index()
varmegye_stats.columns = ['Vármegye', 'Felvételi arány', 'Létszám']
varmegye_stats = varmegye_stats.sort_values('Felvételi arány', ascending=True)

print(varmegye_stats.to_string(index=False))

In [ ]:
# Térképes vizualizáció (ha geopandas elérhető)
try:
    import geopandas as gpd

    gdf = gpd.read_file('../adatok/counties.geojson')

    # Név oszlop azonosítása a GeoJSON-ban
    name_col = None
    for col in gdf.columns:
        if col.lower() in ['name', 'név', 'nev', 'megye', 'varmegye', 'admin_name']:
            name_col = col
            break
    if name_col is None:
        # Az első nem-geometry oszlopot használjuk
        name_col = [c for c in gdf.columns if c != 'geometry'][0]

    print(f'GeoJSON név oszlop: {name_col}')
    print(f'GeoJSON értékek: {sorted(gdf[name_col].unique())}')
    print(f'Train Vármegye értékek: {sorted(train["Vármegye"].unique())}')

    # Összefűzés
    merged = gdf.merge(varmegye_stats, left_on=name_col, right_on='Vármegye', how='left')

    fig, ax = plt.subplots(figsize=(14, 10))
    merged.plot(column='Felvételi arány', ax=ax, legend=True,
                cmap='RdYlGn', edgecolor='black', linewidth=0.5,
                legend_kwds={'label': 'Felvételi arány', 'shrink': 0.6})
    ax.set_title('Felvételi arány vármegyénként', fontsize=16)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f'Térképes vizualizáció nem sikerült ({e}), oszlopdiagram következik:')

    # Fallback oszlopdiagram
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.barh(varmegye_stats['Vármegye'], varmegye_stats['Felvételi arány'],
            color=plt.cm.RdYlGn(varmegye_stats['Felvételi arány']))
    ax.set_xlabel('Felvételi arány')
    ax.set_title('Felvételi arány vármegyénként', fontsize=14)
    ax.axvline(x=train[TARGET].mean(), color='red', linestyle='--',
               label=f'Átlag: {train[TARGET].mean():.3f}')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 2. Előfeldolgozás

In [ ]:
# Eredeti adatok másolata
df_train = train.copy()
df_test = test.copy()

# Célváltozó és ID szétválasztása
y = df_train[TARGET].astype(int)
train_ids = df_train['ID']
test_ids = df_test['ID']

df_train = df_train.drop(columns=[TARGET, 'ID'])
df_test = df_test.drop(columns=['ID'])

print(f'Jellemzők száma: {df_train.shape[1]}')
print(f'Train: {df_train.shape[0]}, Test: {df_test.shape[0]}')

### 2.1 Feature engineering

In [ ]:
def feature_engineering(df):
    """Jellemzők készítése a nyers adatokból."""
    df = df.copy()

    # --- Osztályzatok ---
    osztalyzat_cols = ['Osztályzat_9', 'Osztályzat_10', 'Osztályzat_11', 'Osztályzat_12']
    df['Osztályzat_átlag'] = df[osztalyzat_cols].mean(axis=1)
    df['Osztályzat_trend'] = df['Osztályzat_12'] - df['Osztályzat_9']  # javulás/romlás
    df['Osztályzat_max'] = df[osztalyzat_cols].max(axis=1)
    df['Osztályzat_min'] = df[osztalyzat_cols].min(axis=1)
    df['Osztályzat_szórás'] = df[osztalyzat_cols].std(axis=1)

    # --- Érettségi tantárgyak (ahol -1 = nem vett részt) ---
    tantargyak = ['Történelem', 'Matematika', 'Magyar Nyelv és Irodalom',
                  'Informatika', 'Biológia', 'Fizika', 'Angol', 'Német']

    # Tantárgyak száma (nem -1 értékűek)
    tantargy_mask = df[tantargyak].replace(-1, np.nan)
    df['Tantárgyak_száma'] = tantargy_mask.notna().sum(axis=1)
    df['Tantárgy_átlag'] = tantargy_mask.mean(axis=1)
    df['Tantárgy_max'] = tantargy_mask.max(axis=1)
    df['Tantárgy_min'] = tantargy_mask.min(axis=1)
    df['Tantárgy_szórás'] = tantargy_mask.std(axis=1)

    # --- Emelt szintű érettségik ---
    emelt_cols = ['Informatika_emelt', 'Biológia_emelt', 'Fizika_emelt',
                  'Angol_emelt', 'Német_emelt', 'Matematika_emelt',
                  'Történelem_emelt', 'Magyar Nyelv és Irodalom_emelt']

    # A -1 értékeket 0-nak tekintjük (nem releváns tantárgy)
    emelt_valid = df[emelt_cols].replace(-1, 0)
    df['Emelt_száma'] = emelt_valid.sum(axis=1)

    # --- Kombinált jellemzők ---
    df['Osztályzat_x_Presztízs'] = df['Osztályzat_átlag'] * df['Középiskola Presztízse']
    df['Tanulás_x_Osztályzat'] = df['Tanulási Szokások'] * df['Osztályzat_átlag']
    df['Extra_és_Verseny'] = df['Extrakurrikuláris Tevékenységek'] + df['Versenyeken Való Részvétel']
    df['Ajánlás_és_Munka'] = df['Ajánlások Száma'] + df['Munkatapasztalat']

    # --- Vármegye kódolása (Label Encoding) ---
    # Összes lehetséges vármegye összegyűjtése train és test-ből
    le = LabelEncoder()
    df['Vármegye_kód'] = le.fit_transform(df['Vármegye'].astype(str))

    return df, le


# Train és test együttes feldolgozása a konzisztens kódolásért
df_combined = pd.concat([df_train, df_test], axis=0, ignore_index=True)
df_combined, varmegye_encoder = feature_engineering(df_combined)

# Szétválasztás
df_train_fe = df_combined.iloc[:len(df_train)].copy()
df_test_fe = df_combined.iloc[len(df_train):].copy()

# Vármegye szöveges oszlop eltávolítása
df_train_fe = df_train_fe.drop(columns=['Vármegye'])
df_test_fe = df_test_fe.drop(columns=['Vármegye'])

print(f'Jellemzők száma feature engineering után: {df_train_fe.shape[1]}')
print(f'Új jellemzők: {[c for c in df_train_fe.columns if c not in df_train.columns]}')

### 2.2 -1 értékek kezelése

In [ ]:
# A -1 értékeket meghagyjuk a fa-alapú modelleknél (tudják kezelni),
# de a lineáris modellekhez NaN-ra cseréljük majd.
# Egyelőre készítünk egy jelző oszlopot is.

tantargyak = ['Informatika', 'Biológia', 'Fizika', 'Angol', 'Német']
for col in tantargyak:
    if col in df_train_fe.columns:
        df_train_fe[f'{col}_hiányzik'] = (df_train_fe[col] == -1).astype(int)
        df_test_fe[f'{col}_hiányzik'] = (df_test_fe[col] == -1).astype(int)

print(f'Végső jellemzők száma: {df_train_fe.shape[1]}')
print(df_train_fe.head())

### 2.3 Train/validáció szétválasztás

In [ ]:
X = df_train_fe.values
feature_names = df_train_fe.columns.tolist()
X_test_final = df_test_fe.values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape}, Validáció: {X_val.shape}')
print(f'Train célváltozó eloszlás: {np.mean(y_train):.3f}')
print(f'Validáció célváltozó eloszlás: {np.mean(y_val):.3f}')

## 3. Baseline modell - Logisztikus Regresszió

In [ ]:
# Logisztikus regresszióhoz skálázás szükséges
scaler = StandardScaler()

# NaN-ok kezelése a -1 értékekből (lineáris modellhez)
X_train_lr = pd.DataFrame(X_train, columns=feature_names).copy()
X_val_lr = pd.DataFrame(X_val, columns=feature_names).copy()

# -1 értékeket 0-ra cseréljük a lineáris modellhez
X_train_lr = X_train_lr.replace(-1, 0)
X_val_lr = X_val_lr.replace(-1, 0)

# NaN értékek pótlása 0-val
X_train_lr = X_train_lr.fillna(0)
X_val_lr = X_val_lr.fillna(0)

X_train_scaled = scaler.fit_transform(X_train_lr)
X_val_scaled = scaler.transform(X_val_lr)

# Logisztikus regresszió
lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, C=1.0)
lr_model.fit(X_train_scaled, y_train)

# Előrejelzés és kiértékelés
y_pred_lr = lr_model.predict_proba(X_val_scaled)[:, 1]
auc_lr = roc_auc_score(y_val, y_pred_lr)
print(f'Logisztikus Regresszió ROC-AUC: {auc_lr:.4f}')

In [ ]:
# ROC-görbe
fpr_lr, tpr_lr, _ = roc_curve(y_val, y_pred_lr)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr_lr, tpr_lr, 'b-', linewidth=2,
        label=f'Logisztikus Regresszió (AUC = {auc_lr:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Véletlen (AUC = 0.5)')
ax.set_xlabel('Hamis Pozitív Arány (FPR)')
ax.set_ylabel('Valós Pozitív Arány (TPR)')
ax.set_title('ROC-görbe - Baseline modell', fontsize=14)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Fejlettebb modellek

### 4.1 Random Forest

In [ ]:
# Random Forest - hiperparaméter keresés
rf_params = {
    'n_estimators': [200, 500],
    'max_depth': [8, 12, 16],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4],
}

rf_model = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

rf_grid = GridSearchCV(
    rf_model, rf_params,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

# A fa-alapú modellek tudják kezelni a -1 értékeket, de NaN-t nem
X_train_filled = np.nan_to_num(X_train, nan=0.0)
X_val_filled = np.nan_to_num(X_val, nan=0.0)

rf_grid.fit(X_train_filled, y_train)

print(f'\nLegjobb RF paraméterek: {rf_grid.best_params_}')
print(f'Legjobb RF CV ROC-AUC: {rf_grid.best_score_:.4f}')

y_pred_rf = rf_grid.best_estimator_.predict_proba(X_val_filled)[:, 1]
auc_rf = roc_auc_score(y_val, y_pred_rf)
print(f'RF Validációs ROC-AUC: {auc_rf:.4f}')

### 4.2 XGBoost

In [ ]:
# XGBoost hiperparaméter keresés
xgb_params = {
    'n_estimators': [200, 500],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [1, 5],
}

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    use_label_encoder=False
)

xgb_grid = GridSearchCV(
    xgb_model, xgb_params,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train_filled, y_train)

print(f'\nLegjobb XGBoost paraméterek: {xgb_grid.best_params_}')
print(f'Legjobb XGBoost CV ROC-AUC: {xgb_grid.best_score_:.4f}')

y_pred_xgb = xgb_grid.best_estimator_.predict_proba(X_val_filled)[:, 1]
auc_xgb = roc_auc_score(y_val, y_pred_xgb)
print(f'XGBoost Validációs ROC-AUC: {auc_xgb:.4f}')

### 4.3 LightGBM

In [ ]:
# LightGBM hiperparaméter keresés
lgb_params = {
    'n_estimators': [200, 500, 1000],
    'max_depth': [4, 6, 8, -1],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [31, 63],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [0, 1],
}

lgb_model = lgb.LGBMClassifier(
    objective='binary',
    metric='auc',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

lgb_grid = GridSearchCV(
    lgb_model, lgb_params,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

lgb_grid.fit(X_train_filled, y_train)

print(f'\nLegjobb LightGBM paraméterek: {lgb_grid.best_params_}')
print(f'Legjobb LightGBM CV ROC-AUC: {lgb_grid.best_score_:.4f}')

y_pred_lgb = lgb_grid.best_estimator_.predict_proba(X_val_filled)[:, 1]
auc_lgb = roc_auc_score(y_val, y_pred_lgb)
print(f'LightGBM Validációs ROC-AUC: {auc_lgb:.4f}')

### 4.4 Modellek összehasonlítása

In [ ]:
# Összehasonlító táblázat
results = pd.DataFrame({
    'Modell': ['Logisztikus Regresszió', 'Random Forest', 'XGBoost', 'LightGBM'],
    'Validációs ROC-AUC': [auc_lr, auc_rf, auc_xgb, auc_lgb],
    'CV ROC-AUC': ['-', f'{rf_grid.best_score_:.4f}',
                   f'{xgb_grid.best_score_:.4f}', f'{lgb_grid.best_score_:.4f}']
}).sort_values('Validációs ROC-AUC', ascending=False)

print(results.to_string(index=False))

# ROC-görbék összehasonlítása
fpr_rf, tpr_rf, _ = roc_curve(y_val, y_pred_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_val, y_pred_xgb)
fpr_lgb, tpr_lgb, _ = roc_curve(y_val, y_pred_lgb)

fig, ax = plt.subplots(figsize=(10, 10))
ax.plot(fpr_lr, tpr_lr, linewidth=2, label=f'Logisztikus Regresszió (AUC = {auc_lr:.4f})')
ax.plot(fpr_rf, tpr_rf, linewidth=2, label=f'Random Forest (AUC = {auc_rf:.4f})')
ax.plot(fpr_xgb, tpr_xgb, linewidth=2, label=f'XGBoost (AUC = {auc_xgb:.4f})')
ax.plot(fpr_lgb, tpr_lgb, linewidth=2, label=f'LightGBM (AUC = {auc_lgb:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Véletlen (AUC = 0.5)')
ax.set_xlabel('Hamis Pozitív Arány (FPR)', fontsize=12)
ax.set_ylabel('Valós Pozitív Arány (TPR)', fontsize=12)
ax.set_title('ROC-görbék összehasonlítása', fontsize=16)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Végső modell és submission

### 5.1 Legjobb modell kiválasztása és újratanítása a teljes train adaton

In [ ]:
# A legjobb modell kiválasztása
best_models = {
    'Random Forest': (rf_grid.best_estimator_, auc_rf),
    'XGBoost': (xgb_grid.best_estimator_, auc_xgb),
    'LightGBM': (lgb_grid.best_estimator_, auc_lgb),
}

best_name = max(best_models, key=lambda k: best_models[k][1])
best_model_template = best_models[best_name][0]
print(f'Legjobb modell: {best_name} (AUC = {best_models[best_name][1]:.4f})')
print(f'Paraméterek: {best_model_template.get_params()}')

In [ ]:
# Újratanítás a teljes train adaton
X_full = np.nan_to_num(X, nan=0.0)
X_test_final_filled = np.nan_to_num(X_test_final, nan=0.0)

# Klón a legjobb modellből, teljes adaton tanítás
from sklearn.base import clone
final_model = clone(best_model_template)
final_model.fit(X_full, y)

# Keresztvalidáció a teljes adaton
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(clone(best_model_template), X_full, y,
                            cv=cv, scoring='roc_auc', n_jobs=-1)

print(f'\n5-fold CV ROC-AUC értékek: {cv_scores}')
print(f'Átlagos CV ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

### 5.2 Feature importance

In [ ]:
# Jellemző fontosságok vizualizációja
if hasattr(final_model, 'feature_importances_'):
    importances = final_model.feature_importances_
    feat_imp = pd.DataFrame({
        'Jellemző': feature_names,
        'Fontosság': importances
    }).sort_values('Fontosság', ascending=True)

    # Top 20 jellemző
    top_n = min(20, len(feat_imp))
    feat_imp_top = feat_imp.tail(top_n)

    fig, ax = plt.subplots(figsize=(12, 10))
    bars = ax.barh(feat_imp_top['Jellemző'], feat_imp_top['Fontosság'],
                   color=plt.cm.viridis(np.linspace(0.2, 0.8, top_n)))
    ax.set_xlabel('Fontosság', fontsize=12)
    ax.set_title(f'Top {top_n} legfontosabb jellemző ({best_name})', fontsize=14)
    plt.tight_layout()
    plt.show()

    # Teljes lista
    print('\nÖsszes jellemző fontossága:')
    print(feat_imp.sort_values('Fontosság', ascending=False).to_string(index=False))

### 5.3 Submission fájl generálása

In [ ]:
# Előrejelzés a test adatokon
y_test_pred = final_model.predict(X_test_final_filled)

# Submission DataFrame
submission = pd.DataFrame({
    'ID': test_ids,
    'Felvételi Eredmény': y_test_pred.astype(int)
})

# Ellenőrzés
print(f'Submission méret: {submission.shape}')
print(f'Előrejelzett értékek eloszlása:')
print(submission['Felvételi Eredmény'].value_counts())
print(f'Felvételi arány (előrejelzett): {submission["Felvételi Eredmény"].mean():.3f}')
print(f'Felvételi arány (train): {y.mean():.3f}')

submission.head(10)

In [ ]:
# Submission mentése
submission.to_csv('submission.csv', index=False)
print('submission.csv sikeresen mentve!')
print(f'\n--- Végső eredmény ---')
print(f'Modell: {best_name}')
print(f'5-fold CV ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')